# SNOM Connection & Movement Test
Tests `nea_tools` connection, position read-back, and XY tip movement via the `context.Logic.MoveTipPosition` API.

## 0 — Configuration

In [ ]:
HOST        = 'nea-server'   # change to IP if DNS does not resolve
PATH_TO_DLL = ''             # leave empty unless nea_tools requires a specific path

SPEED_UM_S  = 0.2            # tip movement speed in µm/s

# Test positions in µm — origin is (50, 50) µm
# Visits a small 1 µm square around the default scan origin and returns home.
ORIGIN_UM = (50.0, 50.0)
TEST_POSITIONS_UM = [
    ORIGIN_UM,           # home
    (51.0, 50.0),        # +1 µm X
    (51.0, 51.0),        # +1 µm X, +1 µm Y
    (50.0, 51.0),        # +1 µm Y
    ORIGIN_UM,           # back to origin
]

## 1 — Imports

In [ ]:
import asyncio
from time import sleep

import nest_asyncio
import nea_tools

print('nea_tools version:', getattr(nea_tools, '__version__', 'unknown'))

## 2 — Connect to nea-server

In [ ]:
loop = asyncio.get_event_loop()
nest_asyncio.apply(loop)

print(f'Connecting to {HOST} ...')
loop.run_until_complete(nea_tools.connect(HOST, fingerprint=None, path_to_dll=PATH_TO_DLL))
print('Connected.')

## 3 — Import neaspec and Nea.Client.SharedDefinitions (injected after connect)

In [ ]:
from neaspec import context
import Nea.Client.SharedDefinitions as nea

print('context :', context)
print('nea     :', nea)

## 4 — Read current position

In [ ]:
pos = context.Microscope.GetActiveMotorDistanceToReferenceXyz()
print(f'Current position  X={pos.X:.3f} nm   Y={pos.Y:.3f} nm   Z={pos.Z:.3f} nm')

## 5 — Movement helper
Mirrors the developer snippet exactly: attaches event callbacks, fires `MoveTipPosition`, polls until the move completes.

In [ ]:
def move_tip(x_um: float, y_um: float, speed_um_s: float = SPEED_UM_S) -> None:
    """Move tip to (x_um, y_um) in µm and block until done."""
    do_wait = [False]

    def on_moved(sender, args):
        print(f'  Arrived at {args.Point}')
        do_wait[0] = False

    def on_moving(sender, args):
        print(f'  Moving to {args.Point}')
        do_wait[0] = True

    context.Logic.TipPositionMoved  += on_moved
    context.Logic.TipPositionMoving += on_moving
    try:
        args = nea.MoveTipPositionArgs(
            nea.Geometry.Point2D(x_um, y_um),
            speed_um_s / 1000.0,   # SDK expects µm/ms
        )
        context.Logic.MoveTipPosition.Execute(args)
        sleep(0.1)
        while do_wait[0]:
            sleep(0.1)
        sleep(0.2)   # settling time
    finally:
        context.Logic.TipPositionMoved  -= on_moved
        context.Logic.TipPositionMoving -= on_moving

print('move_tip() defined.')

## 6 — Movement test

In [ ]:
import time

results = []

for (x_um, y_um) in TEST_POSITIONS_UM:
    print(f'\nTarget: ({x_um}, {y_um}) µm')
    t0 = time.time()
    move_tip(x_um, y_um)
    dt = time.time() - t0

    pos = context.Microscope.GetActiveMotorDistanceToReferenceXyz()
    err_x_nm = pos.X - x_um * 1000.0
    err_y_nm = pos.Y - y_um * 1000.0
    print(f'  Readback: X={pos.X:.1f} nm  Y={pos.Y:.1f} nm  '
          f'err=({err_x_nm:+.1f}, {err_y_nm:+.1f}) nm  dt={dt:.2f} s')
    results.append(dict(x_um=x_um, y_um=y_um,
                        readback_x_nm=pos.X, readback_y_nm=pos.Y,
                        err_x_nm=err_x_nm, err_y_nm=err_y_nm, dt_s=dt))

print('\nMovement test complete.')

## 7 — Summary table

In [ ]:
import pandas as pd
df = pd.DataFrame(results)
df

## 8 — Read SNOM signals at final position

In [ ]:
mic = context.Microscope.Py

print(f'{'Harmonic':>10}  {'O_Amp':>12}  {'M_Amp':>12}')
print('-' * 40)
for h in range(6):
    try:
        oa = float(mic.OpticalAmplitude(h))
    except Exception as e:
        oa = float('nan')
    try:
        ma = float(mic.MechanicalAmplitude(h))
    except Exception as e:
        ma = float('nan')
    print(f'{h:>10}  {oa:>12.6f}  {ma:>12.6f}')

## 9 — Disconnect

In [ ]:
try:
    nea_tools.disconnect()
    print('Disconnected.')
except Exception as e:
    print(f'Disconnect error (ignored): {e}')